In [3]:
# =============================================
# NOTEBOOK SETUP — FINAL WORKING VERSION
# =============================================
import sys
from pathlib import Path

# Reliable project root (works from inside notebooks/)
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import duckdb
from config.settings import DB_PATH

print(f"✅ Project root: {PROJECT_ROOT}")
print(f"✅ DB path: {DB_PATH}")

# Connect writable
con = duckdb.connect(str(DB_PATH))
print("✅ Connected in writable mode!")

# Create view using FULL ABSOLUTE PATH (this is what was breaking)
con.sql(f"""
    CREATE OR REPLACE VIEW silver_lineups AS 
    SELECT * 
    FROM read_parquet('{PROJECT_ROOT}/data/silver/lineups/year=*/month=*/day=*/lineups.parquet', 
                      hive_partitioning=1)
""")

print("✅ silver_lineups view created with correct absolute path!")

# Test the count
total_df = con.sql("SELECT COUNT(*) as total FROM silver_lineups").df()
total = total_df.iloc[0]['total']
print(f"✅ Total games in silver_lineups: {total:,}")

# Switch to read-only for the rest of the notebook
con.close()
con = duckdb.connect(str(DB_PATH), read_only=True)
print("✅ Switched to read-only mode — you're good to go!")

✅ Project root: /Users/patrickmaloney/Documents/mlb-betting
✅ DB path: /Users/patrickmaloney/Documents/mlb-betting/data/db/mlb_betting.duckdb
✅ Connected successfully!

Existing tables/views:
Empty DataFrame
Columns: [name]
Index: []
✅ Ready — paste the terminal output above so we can fix the silver view next!


In [4]:
# Most recent lineups with the new bbref columns
con.sql("""
    SELECT 
        game_date,
        away_team,
        home_team,
        away_starter_name,
        away_starter_primary_bbref_id,
        away_lineup_bbref_ids[0:3] AS first_three_away_bbref_ids,
        is_confirmed
    FROM silver_lineups 
    ORDER BY fetch_timestamp DESC 
    LIMIT 5
""").df()

ConnectionException: Connection Error: Connection already closed!

In [ ]:
# Quick check that matching worked
con.sql("""
    SELECT 
        game_date,
        away_team,
        away_starter_name,
        away_starter_primary_bbref_id
    FROM silver_lineups 
    WHERE away_starter_primary_bbref_id IS NOT NULL
    LIMIT 8
""").df()


In [ ]:
# Quick validation — how many players matched?
con.sql("""
    SELECT 
        COUNT(CASE WHEN away_starter_primary_bbref_id IS NOT NULL THEN 1 END) as starters_matched,
        COUNT(*) as total_starters
    FROM silver_lineups
""").df()

In [ ]:
# Most recent lineups with the new bbref columns
x = con.sql("""
    SELECT *
    FROM silver_lineups 
    ORDER BY fetch_timestamp DESC 
    LIMIT 5
""").df()
x